# Stage 3 exercise — Attention from scratch

**Goal:** implement the core math behind every transformer with nothing but numpy.
Complete the 3 TODO functions, then run the `# ✅ TEST` cells until they all pass.

No NBA data here — just toy tensors. This is the mechanism that, later, powers the
text-to-SQL step of Statlas.

Reference while you work: the Illustrated Transformer + *Attention Is All You Need* §3.2.

Rule: don't peek at the test cells for the formula — write your version first, then let the
tests check it.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

## Exercise 1 — `softmax`

Turn a vector of scores into a probability distribution along the **last axis**.
Make it **numerically stable** (subtract the row max before exponentiating, so large
inputs don't overflow). Must work on a 1-D vector and on each row of a 2-D matrix.

In [ ]:
def softmax(x):
    """Stable softmax over the last axis. Input: np.ndarray (1-D or 2-D). Output: same shape."""
    # TODO: implement (hint: subtract x.max(axis=-1, keepdims=True), exp, divide by sum)
    raise NotImplementedError

In [ ]:
# ✅ TEST softmax
a = softmax(np.array([1.0, 2.0, 3.0]))
assert np.allclose(a, [0.09003057, 0.24472847, 0.66524096]), a
M = softmax(np.array([[1.0, 2.0, 3.0], [1.0, 1.0, 1.0]]))
assert np.allclose(M.sum(axis=-1), [1.0, 1.0]), 'rows must sum to 1'
assert np.allclose(M[1], [1/3, 1/3, 1/3]), 'equal inputs -> uniform'
big = softmax(np.array([1000.0, 1001.0, 1002.0]))  # must not overflow to nan
assert not np.isnan(big).any(), 'not numerically stable'
print('softmax ok ✅')

## Exercise 2 — `scaled_dot_product_attention`

The heart of it. Given queries `Q` (n_q × d), keys `K` (n_k × d), values `V` (n_k × d_v):

1. scores = Q @ Kᵀ
2. scale by 1/√d  (d = key dimension)
3. if `mask` is given, **add** it to the scores (it carries 0 for allowed, big-negative for blocked)
4. weights = softmax(scores) over the last axis
5. output = weights @ V

Return `(output, weights)`.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Return (output, weights). Q:(n_q,d) K:(n_k,d) V:(n_k,d_v) mask:(n_q,n_k) additive or None."""
    # TODO: implement the 5 steps above (reuse your softmax)
    raise NotImplementedError

In [ ]:
# ✅ TEST scaled_dot_product_attention (invariants — no formula given away)
rng = np.random.default_rng(0)
Q, K, V = rng.normal(size=(3, 4)), rng.normal(size=(5, 4)), rng.normal(size=(5, 6))
out, w = scaled_dot_product_attention(Q, K, V)
assert out.shape == (3, 6), out.shape           # one output row per query, width of V
assert w.shape == (3, 5), w.shape               # weights: queries x keys
assert np.allclose(w.sum(axis=-1), 1.0), 'each query\'s weights must sum to 1'
assert (w >= 0).all(), 'weights must be non-negative'
# If all keys are identical, attention can\'t prefer any -> output = mean of V
Kc = np.ones((5, 4)); out2, w2 = scaled_dot_product_attention(np.ones((1, 4)), Kc, V)
assert np.allclose(out2[0], V.mean(axis=0)), 'uniform attention should average V'
print('attention ok ✅')

## Exercise 3 — `causal_mask`

In a GPT, token *i* may only attend to tokens *j ≤ i* (no peeking at the future).
Return an `(n, n)` **additive** mask: `0.0` where attention is allowed (j ≤ i) and a large
negative number (use `-1e9`) where it's blocked (j > i, the strict upper triangle).

In [ ]:
def causal_mask(n):
    """Return (n,n) additive causal mask: 0 on/below diagonal, -1e9 above."""
    # TODO: implement (hint: np.triu with k=1 marks the strictly-upper triangle)
    raise NotImplementedError

In [ ]:
# ✅ TEST causal_mask + masked attention
m = causal_mask(4)
assert m.shape == (4, 4)
assert np.allclose(np.tril(m), 0.0), 'lower triangle + diagonal must be 0 (allowed)'
assert (m[np.triu_indices(4, k=1)] < -1e8).all(), 'strict upper triangle must be blocked'
# With the mask, query 0 should put ZERO weight on future keys
Q = K = V = rng.normal(size=(4, 4))
out, w = scaled_dot_product_attention(Q, K, V, mask=causal_mask(4))
assert np.allclose(np.triu(w, k=1), 0.0), 'no attention should leak to the future'
assert np.allclose(w.sum(axis=-1), 1.0)
print('causal mask ok ✅  — all done, nice work!')

## Reflection (write 2-3 sentences)

- Why divide the scores by √d?  (what goes wrong for large d if you don't?)
- In your own words: what do Q, K, and V each represent?
- How does the causal mask turn this into a *language model* you can generate from?

_Your notes:_
